In [19]:
import glob

# LINUX-based system
base_path = "/home/rafay/Documents/rafay/WSL_and_Stuff/other/open_source/CenterSnap"
calib_dataset_name = "utils"
annotation_dir = "chessboard_corners"
checkerboardsize = (9, 7)
images = glob.glob("calib_data/*.png")
annotation_dir = "chessboard_corners"
images


['calib_data/67_color.png',
 'calib_data/10_color.png',
 'calib_data/3_color.png',
 'calib_data/11_color.png',
 'calib_data/58_color.png',
 'calib_data/80_color.png',
 'calib_data/41_color.png',
 'calib_data/42_color.png',
 'calib_data/2_color.png',
 'calib_data/15_color.png',
 'calib_data/40_color.png',
 'calib_data/32_color.png',
 'calib_data/28_color.png',
 'calib_data/39_color.png',
 'calib_data/76_color.png',
 'calib_data/70_color.png',
 'calib_data/7_color.png',
 'calib_data/20_color.png',
 'calib_data/63_color.png',
 'calib_data/23_color.png',
 'calib_data/24_color.png',
 'calib_data/81_color.png',
 'calib_data/8_color.png',
 'calib_data/86_color.png',
 'calib_data/34_color.png',
 'calib_data/64_color.png',
 'calib_data/38_color.png',
 'calib_data/54_color.png',
 'calib_data/0_color.png',
 'calib_data/77_color.png',
 'calib_data/69_color.png',
 'calib_data/48_color.png',
 'calib_data/59_color.png',
 'calib_data/21_color.png',
 'calib_data/52_color.png',
 'calib_data/53_color.png

In [20]:
# !pip3 install numpy
# !pip3 install opencv-python
# !pip3 install open3d
# !pip3 install matplotlib

import numpy as np
import cv2
import os
import open3d as o3d
import matplotlib.pyplot as plt

# 1. Camera Calibration
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
square_size = 0.015  #15mm squares
objp = np.zeros((checkerboardsize[0] * checkerboardsize[1], 3), np.float32)
objp[:, :2] = square_size*np.mgrid[0 : checkerboardsize[0], 0 : checkerboardsize[1]].T.reshape(
    -1, 2
)

objpoints = []
imgpoints = []
images = sorted([os.path.join(base_path, calib_dataset_name, img_path) for img_path in images])
save_chessboard_corner_ann = False
if not os.path.exists(annotation_dir) and save_chessboard_corner_ann:
    os.mkdir(annotation_dir)

for fname in images:
    print(fname)
    img = cv2.imread(fname)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    ret, corners = cv2.findChessboardCorners(
        gray,
        checkerboardsize,
        cv2.CALIB_CB_ADAPTIVE_THRESH
        + cv2.CALIB_CB_FAST_CHECK
        + cv2.CALIB_CB_NORMALIZE_IMAGE,
    )

    if ret == True:
        objpoints.append(objp)
        corners2 = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
        imgpoints.append(corners2)
        if save_chessboard_corner_ann:
            fname_ = (
                os.path.join(base_path, annotation_dir)
                + "/"
                + fname.split("/")[-1].split(".")[0]
                + "corner_plot.jpg"
            )
            cv2.drawChessboardCorners(img, checkerboardsize, corners2, ret)
            cv2.imwrite(fname_, img)

ret, cameraMatrix, dist, rvec, tvec = cv2.calibrateCamera(
    objpoints, imgpoints, gray.shape[::-1], None, None
)

# Compute reprojection error
reprojection_error = 0
for i in range(len(objpoints)):
    imgpoints2, _ = cv2.projectPoints(
        objpoints[i], rvec[i], tvec[i], cameraMatrix, dist
    )
    error = cv2.norm(imgpoints[i], imgpoints2, cv2.NORM_L2) / len(imgpoints2)
    reprojection_error += error
reprojection_error /= len(objpoints)
print("Reprojection error:", reprojection_error)
print("Camera Calibrated:   ", ret)
print("Camera Matrix:   ", cameraMatrix)
print("Distortion Parameters:   ", dist)
print("Rotation Vector: ", rvec)
print("Translation Vector: ", tvec)

/home/rafay/Documents/rafay/WSL_and_Stuff/other/open_source/CenterSnap/utils/calib_data/0_color.png
/home/rafay/Documents/rafay/WSL_and_Stuff/other/open_source/CenterSnap/utils/calib_data/10_color.png
/home/rafay/Documents/rafay/WSL_and_Stuff/other/open_source/CenterSnap/utils/calib_data/11_color.png
/home/rafay/Documents/rafay/WSL_and_Stuff/other/open_source/CenterSnap/utils/calib_data/12_color.png
/home/rafay/Documents/rafay/WSL_and_Stuff/other/open_source/CenterSnap/utils/calib_data/13_color.png
/home/rafay/Documents/rafay/WSL_and_Stuff/other/open_source/CenterSnap/utils/calib_data/14_color.png
/home/rafay/Documents/rafay/WSL_and_Stuff/other/open_source/CenterSnap/utils/calib_data/15_color.png
/home/rafay/Documents/rafay/WSL_and_Stuff/other/open_source/CenterSnap/utils/calib_data/16_color.png
/home/rafay/Documents/rafay/WSL_and_Stuff/other/open_source/CenterSnap/utils/calib_data/17_color.png
/home/rafay/Documents/rafay/WSL_and_Stuff/other/open_source/CenterSnap/utils/calib_data/18_c

In [5]:
import numpy as np
import cv2
import glob
import argparse
import os

square_size = 0.015  #15mm squares

class StereoCalibration(object):
    def __init__(self, filepath):
        # termination criteria
        self.criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
        self.criteria_cal = (
            cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER,
            100,
            1e-5,
        )

        # prepare object points, like (0,0,0), (1,0,0), (2,0,0) ....,(6,5,0)
        self.objp = np.zeros((9 * 7, 3), np.float32)
        self.objp[:, :2] = square_size*np.mgrid[0:9, 0:7].T.reshape(-1, 2)

        # Arrays to store object points and image points from all the images.
        self.objpoints = []  # 3d point in real world space
        self.imgpoints_l = []  # 2d points in image plane.
        self.imgpoints_r = []  # 2d points in image plane.

        self.cal_path = filepath
        self.read_images(self.cal_path)

    def read_images(self, cal_path):
        images_right = sorted([os.path.join(cal_path, img_path) for img_path in images])
        images_left = sorted([os.path.join(cal_path, img_path) for img_path in images])

        for i, fname in enumerate(images_right):
            img_l = cv2.imread(images_left[i])
            img_r = cv2.imread(images_right[i])

            gray_l = cv2.cvtColor(img_l, cv2.COLOR_BGR2GRAY)
            gray_r = cv2.cvtColor(img_r, cv2.COLOR_BGR2GRAY)

            # Find the chess board corners
            ret_l, corners_l = cv2.findChessboardCorners(gray_l, checkerboardsize, cv2.CALIB_CB_ADAPTIVE_THRESH
        + cv2.CALIB_CB_FAST_CHECK
        + cv2.CALIB_CB_NORMALIZE_IMAGE)
            ret_r, corners_r = cv2.findChessboardCorners(gray_r, checkerboardsize, cv2.CALIB_CB_ADAPTIVE_THRESH
        + cv2.CALIB_CB_FAST_CHECK
        + cv2.CALIB_CB_NORMALIZE_IMAGE)

            # If found, add object points, image points (after refining them)
            self.objpoints.append(self.objp)

            if ret_l is True:
                rt = cv2.cornerSubPix(
                    gray_l, corners_l, (11, 11), (-1, -1), self.criteria
                )
                self.imgpoints_l.append(corners_l)

                # Draw and display the corners
                ret_l = cv2.drawChessboardCorners(img_l, checkerboardsize, corners_l, ret_l)

            if ret_r is True:
                rt = cv2.cornerSubPix(
                    gray_r, corners_r, (11, 11), (-1, -1), self.criteria
                )
                self.imgpoints_r.append(corners_r)

                # Draw and display the corners
                ret_r = cv2.drawChessboardCorners(img_r, checkerboardsize, corners_r, ret_r)
        img_shape = gray_l.shape[::-1]

        rt, self.M1, self.d1, self.r1, self.t1 = cv2.calibrateCamera(
            self.objpoints, self.imgpoints_l, img_shape, None, None
        )
        rt, self.M2, self.d2, self.r2, self.t2 = cv2.calibrateCamera(
            self.objpoints, self.imgpoints_r, img_shape, None, None
        )

        self.camera_model = self.stereo_calibrate(img_shape)

    def stereo_calibrate(self, dims):
        flags = 0
        flags |= cv2.CALIB_FIX_INTRINSIC
        flags |= cv2.CALIB_USE_INTRINSIC_GUESS
        flags |= cv2.CALIB_FIX_FOCAL_LENGTH
        flags |= cv2.CALIB_ZERO_TANGENT_DIST

        stereocalib_criteria = (
            cv2.TERM_CRITERIA_MAX_ITER + cv2.TERM_CRITERIA_EPS,
            100,
            1e-5,
        )
        ret, M1, d1, M2, d2, R, T, E, F = cv2.stereoCalibrate(
            self.objpoints,
            self.imgpoints_l,
            self.imgpoints_r,
            self.M1,
            self.d1,
            self.M2,
            self.d2,
            dims,
            criteria=stereocalib_criteria,
            flags=flags,
        )

        print("Intrinsic_mtx_1", M1)
        print("dist_1", d1)
        print("Intrinsic_mtx_2", M2)
        print("dist_2", d2)
        print("R", R)
        print("T", T)
        print("E", E)
        print("F", F)
        camera_model = dict(
            [
                ("M1", M1),
                ("M2", M2),
                ("dist1", d1),
                ("dist2", d2),
                ("rvecs1", self.r1),
                ("rvecs2", self.r2),
                ("R", R),
                ("T", T),
                ("E", E),
                ("F", F),
            ]
        )

        return camera_model

In [6]:
cal_data = StereoCalibration(base_path)

Intrinsic_mtx_1 [[611.03066846   0.         345.46594134]
 [  0.         609.2503304  245.81101883]
 [  0.           0.           1.        ]]
dist_1 [[ 0.05926032  0.32171581 -0.00162027  0.01252162 -1.29607388]]
Intrinsic_mtx_2 [[611.03066846   0.         345.46594134]
 [  0.         609.2503304  245.81101883]
 [  0.           0.           1.        ]]
dist_2 [[ 0.05926032  0.32171581 -0.00162027  0.01252162 -1.29607388]]
R [[ 1.00000000e+00 -8.68334627e-14  6.39445599e-14]
 [ 8.68334627e-14  1.00000000e+00 -1.57748866e-13]
 [-6.39445599e-14  1.57748866e-13  1.00000000e+00]]
T [[-2.26147966e-14]
 [ 6.05087429e-14]
 [ 2.62721562e-14]]
E [[-6.15050723e-27 -2.62721562e-14  6.05087429e-14]
 [ 2.62721562e-14  1.28615622e-27  2.26147966e-14]
 [-6.05087429e-14 -2.26147966e-14 -3.01746426e-28]]
F [[ 3.88441142e-06  1.66409165e+07 -2.74409800e+10]
 [-1.66409165e+07 -8.17038423e-07 -3.00373289e+09]
 [ 2.74409800e+10  3.00373289e+09  1.00000000e+00]]


In [18]:
# # Convert the first rvec to a rotation matrix
# rotation_matrix, _ = cv2.Rodrigues(cal_data.camera_model['rvecs1'][2])
# print("Rotation Matrix:\n", rotation_matrix)